In [1]:
import os, subprocess

REPO = "/kaggle/working/atml_internal"
if not os.path.exists(REPO):
    r = subprocess.run(
        ["git", "clone", "https://github.com/rana-taqveem/atml_internal.git", REPO],
        capture_output=True, text=True)
    print(r.stdout, r.stderr)

assert os.path.exists(f"{REPO}/assignment_01/task4/scripts/train.py"), \
    "Clone failed — is Internet enabled for this version?"
%cd {REPO}
print("repo ready")


 Cloning into '/kaggle/working/atml_internal'...
Updating files:  42% (123/291)
Updating files:  43% (126/291)
Updating files:  44% (129/291)
Updating files:  45% (131/291)
Updating files:  46% (134/291)
Updating files:  47% (137/291)
Updating files:  48% (140/291)
Updating files:  49% (143/291)
Updating files:  50% (146/291)
Updating files:  51% (149/291)
Updating files:  52% (152/291)
Updating files:  53% (155/291)
Updating files:  54% (158/291)
Updating files:  55% (161/291)
Updating files:  56% (163/291)
Updating files:  57% (166/291)
Updating files:  58% (169/291)
Updating files:  59% (172/291)
Updating files:  60% (175/291)
Updating files:  61% (178/291)
Updating files:  62% (181/291)
Updating files:  63% (184/291)
Updating files:  64% (187/291)
Updating files:  65% (190/291)
Updating files:  66% (193/291)
Updating files:  67% (195/291)
Updating files:  68% (198/291)
Updating files:  69% (201/291)
Updating files:  70% (204/291)
Updating files:  71% (207/291)
Updating files:  72% 

In [2]:
# Cell 2 — confirm GPU and see what the uploaded dataset is actually called
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


CUDA: True Tesla T4


In [3]:
!cd /kaggle/working/atml_internal && git pull


Already up to date.


In [4]:
import os, shutil, glob

SRC = "/kaggle/input/datasets/ranataqveemulhassan/task4-dataset"
DST = "/kaggle/working/atml/task4/dataset"
os.makedirs(DST, exist_ok=True)

# --- CIFAR-10: torchvision requires the folder to be named cifar-10-batches-py
src10 = f"{SRC}/cifar-10-python"
inner = glob.glob(f"{src10}/**/data_batch_1", recursive=True)
if not inner:
    raise SystemExit(f"data_batch_1 not found under {src10}; contents: {os.listdir(src10)}")
batches_dir = os.path.dirname(inner[0])
target10 = f"{DST}/cifar-10-batches-py"
if not os.path.exists(target10):
    shutil.copytree(batches_dir, target10)
print("CIFAR-10 ->", target10, sorted(os.listdir(target10))[:4])

# --- CIFAR-100: extracted folder is already named correctly
src100 = f"{SRC}/cifar-100-python"
meta = glob.glob(f"{src100}/**/meta", recursive=True)
if not meta:
    raise SystemExit(f"meta not found under {src100}; contents: {os.listdir(src100)}")
py_dir = os.path.dirname(meta[0])
target100 = f"{DST}/cifar-100-python"
if not os.path.exists(target100):
    shutil.copytree(py_dir, target100)
print("CIFAR-100 ->", target100, sorted(os.listdir(target100)))


CIFAR-10 -> /kaggle/working/atml/task4/dataset/cifar-10-batches-py ['batches.meta', 'data_batch_1', 'data_batch_2', 'data_batch_3']
CIFAR-100 -> /kaggle/working/atml/task4/dataset/cifar-100-python ['file.txt~', 'meta', 'test', 'train']


In [5]:
from torchvision.datasets import CIFAR10, CIFAR100
R = "/kaggle/working/atml/task4/dataset"
print("CIFAR-10 train:", len(CIFAR10(root=R, train=True,  download=False)))
print("CIFAR-10 test :", len(CIFAR10(root=R, train=False, download=False)))
print("CIFAR-100 test:", len(CIFAR100(root=R, train=False, download=False)))


CIFAR-10 train: 50000
CIFAR-10 test : 10000
CIFAR-100 test: 10000


In [6]:
# Cell A — training
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.train --method vanilla
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.train --method gcsc
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.train --method proser


Initialized directories for task: task4
Wrote CIFAR-10 split: 45000 train / 5000 val -> /kaggle/working/atml/task4/dataset/cifar10_split_seed6304.json
CIFAR-10: 45000 train / 5000 val / 10000 test

Training vanilla: 100 epochs, 352 steps each, SGD lr 0.1 cosine
Epoch 1/100 | loss 2.0527, train acc 26.44% | val acc 37.36%  new best | 36s, ~59 min left
Epoch 2/100 | loss 1.4821, train acc 45.18% | val acc 50.28%  new best | 36s, ~58 min left
Epoch 3/100 | loss 1.2052, train acc 56.41% | val acc 58.48%  new best | 36s, ~59 min left
Epoch 4/100 | loss 0.9916, train acc 64.62% | val acc 66.82%  new best | 37s, ~60 min left
Epoch 5/100 | loss 0.8237, train acc 71.23% | val acc 67.14%  new best | 38s, ~60 min left
Epoch 6/100 | loss 0.7072, train acc 75.28% | val acc 76.84%  new best | 39s, ~62 min left
Epoch 7/100 | loss 0.6295, train acc 78.35% | val acc 73.66% | 39s, ~61 min left
Epoch 8/100 | loss 0.5616, train acc 80.59% | val acc 70.04% | 39s, ~60 min left
Epoch 9/100 | loss 0.5317, tra

In [7]:
# Cell B — CIFAR-100 enters here, after all checkpoints are fixed
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.extract_outputs --method vanilla
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.extract_outputs --method gcsc
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.extract_outputs --method proser


Initialized directories for task: task4
CIFAR-10: 45000 train / 5000 val / 10000 test
near unknowns: 800 images across 8 CIFAR-100 classes
far unknowns: 800 images across 8 CIFAR-100 classes

Extracting outputs for vanilla ...
   train (45000, 512), val (5000, 512), test (10000, 512)
   near (800, 512), far (800, 512)
   logits 10 columns
Saved /kaggle/working/atml/task4/cache/vanilla_outputs.npz
Initialized directories for task: task4
CIFAR-10: 45000 train / 5000 val / 10000 test
near unknowns: 800 images across 8 CIFAR-100 classes
far unknowns: 800 images across 8 CIFAR-100 classes

Extracting outputs for gcsc ...
   train (45000, 512), val (5000, 512), test (10000, 512)
   near (800, 512), far (800, 512)
   logits 10 columns
Saved /kaggle/working/atml/task4/cache/gcsc_outputs.npz
Initialized directories for task: task4
CIFAR-10: 45000 train / 5000 val / 10000 test
near unknowns: 800 images across 8 CIFAR-100 classes
far unknowns: 800 images across 8 CIFAR-100 classes

Extracting out

In [8]:
# Cell C — tables, failures, figure
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.evaluate_osr
!cd /kaggle/working/atml_internal && python -m assignment_01.task4.scripts.make_figures


Initialized directories for task: task4

=== Table 1: post-hoc scores on the frozen Vanilla model
score         AUROC near  AUROC far  AUROC all  known acc  rej near  rej far
msp               0.8139     0.8867     0.8503     94.09%    31.37%   46.38%
mls               0.8022     0.8825     0.8424     94.22%    34.75%   54.12%
energy            0.8024     0.8830     0.8427     94.41%    34.50%   53.25%
mahalanobis       0.7952     0.9106     0.8529     94.36%    27.75%   51.37%

=== Table 2: Vanilla / GCSC / PROSER (MLS common score)
model         score          csa_pct  AUROC near  AUROC far  AUROC all  known acc  rej near  rej far
vanilla       mls              94.54      0.8022     0.8825     0.8424     94.22%    34.75%   54.12%
gcsc          mls              95.12      0.8265     0.9092     0.8679     94.60%    37.12%   59.00%
proser        mls              94.63      0.8031     0.8869     0.8450     94.64%    32.75%   50.00%
proser        placeholder      94.63      0.7796     0.8

In [9]:
# Cell D — collect outputs
!cp -r /kaggle/working/atml/task4/results /kaggle/working/task4_results
!ls -la /kaggle/working/task4_results


total 772
drwxr-xr-x 2 root root   4096 Sep 24 00:00 .
drwxr-xr-x 5 root root   4096 Sep 24 00:00 ..
-rw-r--r-- 1 root root   2112 Sep 24 00:00 failure_analysis.csv
-rw-r--r-- 1 root root 721180 Sep 24 00:00 figure_scores.json
-rw-r--r-- 1 root root  10821 Sep 24 00:00 gcsc_history.json
-rw-r--r-- 1 root root  11407 Sep 24 00:00 osr_results.json
-rw-r--r-- 1 root root   6800 Sep 24 00:00 proser_history.json
-rw-r--r-- 1 root root    800 Sep 24 00:00 table1_posthoc_scores.csv
-rw-r--r-- 1 root root    878 Sep 24 00:00 table2_model_comparison.csv
-rw-r--r-- 1 root root  10811 Sep 24 00:00 vanilla_history.json
